## Point 8. Baselines: RAG over KG vs. Zero-Shot LLM

- RAG-KG: Retrieve from KG (SPARQL/Cypher) + chunk text; feed to an LLM to generate answers with
citations.
- Zero-shot LLM: Prompt a non-fine-tuned model using only the user instruction (no KG).
- Record answers and metadata (latency, context length).

In [1]:
import json
import time
from pathlib import Path
import pandas as pd

# Load facts
facts = []
with open("data/facts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        facts.append(json.loads(line))

facts_df = pd.DataFrame(facts)
print(f"Loaded {len(facts_df)} facts from KG")

# Load raw text for context retrieval
raw_texts = []
if Path("data/raw_text.jsonl").exists():
    with open("data/raw_text.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            raw_texts.append(json.loads(line))
    raw_text_df = pd.DataFrame(raw_texts)
    print(f"Loaded {len(raw_text_df)} text chunks")

Loaded 2247 facts from KG
Loaded 134 text chunks


In [2]:
def predicate_to_verb(predicate):
    mapping = {
        "hasNutrient": "contains",
        "usesTechnique": "is prepared using",
        "hasGuideline": "follows the guideline",
        "recommendsTechnique": "recommends",
        "aimsToImprove": "aims to improve",
        "affectsRiskOf": "affects the risk of",
        "associatedWithOutcome": "is associated with",
        "hasEnvironmentalImpact": "has an environmental impact",
        "guidelineTargetsImpact": "targets environmental impact",
        "affectsImpactCategory": "affects the environmental impact category"
    }
    return mapping.get(predicate, predicate)

In [3]:
term_conversion = {
        "nutrient" : ["hasNutrient"],
        "contain" : ["hasNutrient"],
        "use" : ["usesTechnique", "hasNutrient"],
        "rich" : ["hasNutrient"],
        "find" : ["hasNutrient"],
        "ingredient" : ["hasNutrient"],
        "follow" : ["usesTechnique", "hasGuideline"],
        "affect" : ["hasEnvironmentalImpact", "affectsRiskOf", "guidelineTargetsImpact", "affectsImpactCategory"],
        "risk" : ["affectRiskOf"],
        "associate" : ["associatedWithOutcome", "hasNutrient", "usesTechnique"],
        "outcome" : ["associatedWithOutcome"],
        "recommend" : ["recommendsTechnique"],
        "environment" : ["hasEnvironmentalImpact", "guidelineTargetsImpact", "affectsImpactCategory"],
        "impact" : ["hasEnvironmentalImpact", "affectsRiskOf", "guidelineTargetsImpact", "affectsImpactCategory"],
        "prepare" : ["usesTechnique", "hasGuideline"],
        "guideline" : ["hasGuideline", "guidelineTargetsImpact"],
        "aim" : ["aimsToImprove", "hasGuideline", "guidelineTargetsImpact"],
        "improve" : ["aimsToImprove", "hasGuideline"],
        "target" : ["guidelineTargetsImpact"],
        "sustain" : ["guidelineTargetsImpact", "aimsToImprove", "hasEnvironmentalImpact", "affectsImpactCategory"],
        "benefit" : ["usesTechnique", "hasNutrient", "aimsToImprove", "hasGuideline", "hasEnvironmentalImpact", "affectsRiskOf", "guidelineTargetsImpact", "affectsImpactCategory"],
        "dietary": ["hasNutrient", "hasGuideline"],
        "advice": ["hasGuideline"],
        "preparation": ["usesTechnique", "hasNutrient"],
        "method": ["usesTechnique", "associatedWithOutcome"],
        "cook": ["usesTechnique", "associatedWithOutcome", "hasEnvironmentalImpact"],
        "process": ["usesTechnique", "hasNutrient", "associatedWithOutcome"]
    }

def predicate_to_search(query_terms):
    pred = []
    for term in query_terms:
        if term in term_conversion.keys():
            for p in term_conversion[term]:
                if p not in pred:
                    pred.append(p)
    return pred

In [4]:
from Levenshtein import distance as levenshtein
import spacy
nlp = spacy.load("en_core_web_sm")

def fuzzy_match(df, term, predicate, edit=True):
    """Return rows where subject/object matches term exactly or within edit distance ≤ 1."""
    
    def check_row(row):
        subj = str(row['subject']).lower()
        obj = str(row['object']).lower()
        t   = term.lower()

        # Exact match first (fast)
        if subj == t or obj == t:
            return True
        if edit: 
            # Fuzzy match (edit distance ≤ 1)
            if levenshtein(subj, t) <= 1 or levenshtein(obj, t) <= 1:
                return True
        
        return False

    # Filter rows by predicate + fuzzy function
    filtered = df[df['predicate'] == predicate]
    return filtered[filtered.apply(check_row, axis=1)]


def retrieve_from_kg(unigram, bigram = None):
    """
    Retrieve relevant facts from KG based on query terms.
    Uses simple keyword matching across subject, predicate, object.
    """
    results = []
    pred = predicate_to_search(unigram)
    for p in pred:
        if bigram:
            for terms in bigram:
                # print(term)
                # Search in subject, predicate, or object
                matches = fuzzy_match(facts_df, terms, p, edit=False) ## Messo False per i bigram altrimenti dà problemi con tutte le vitamine
                if not matches.empty:
                    token_remove = terms.split("_")
                    unigram = [u for u in unigram if u not in token_remove]
                # print(f"TERM: {term}\n MATCHES:{matches}")
                results.append(matches)
        if unigram:    
            for term in unigram:
                # Search in subject, predicate, or object
                matches = fuzzy_match(facts_df, term, p)
                results.append(matches)
    # print(results)
    if results:
        combined = pd.concat(results)
        combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
        combined = combined.drop_duplicates()
        return combined

    return pd.DataFrame()


def get_text_context(fact_pages, top_k=100):
    """
    Retrieve original text chunks associated with the pages in retrieved facts.
    """
    if not raw_texts:
        return []
    
    context_chunks = []
    for page in fact_pages:
        chunks = [t for t in raw_texts if t.get('page') == page]
        context_chunks.extend(chunks)#[:top_k])
    
    return context_chunks[:top_k]


def extract_query_terms(question):
    """
    Extraction of key terms from question.
    """

    doc = nlp(question.lower())

    unigram = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]

    bigram = ["_".join(unigram[i:i+2]) for i in range(len(unigram)-1)]
    
    return unigram, bigram 

print("KG retrieval functions defined")

c:\Users\salir\Desktop\Oulu\NLP\env310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KG retrieval functions defined


In [5]:
import os
from groq import Groq

# Initialize client
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

In [6]:
def rag_kg_baseline(question, model_name="groq/compound", use_chunks=False, pred_conversion=True):
    """
    RAG over Knowledge Graph baseline.
    Retrieves facts from KG, builds context, and generates answer with LLM.
    
    Returns:
        - answer: Generated text
        - metadata: dict with context, facts, latency, context_length
    """
    start_time = time.time()
    
    # 1. Extract query terms and retrieve from KG
    unigram, bigram = extract_query_terms(question)
    #print(unigram, bigram)
    retrieved_facts = retrieve_from_kg(unigram, bigram)
    
    # 2. Build context from retrieved facts
    context_parts = []
    pages_cited = []
    contexts = {}

    if not retrieved_facts.empty:
        context_parts.append("Relevant Knowledge Graph Facts:")
        for idx, row in retrieved_facts.iterrows():
            key = (row['subject'], row['predicate'], row['object'])
            page = row.get('page')
            if key not in contexts:
                contexts[key] = []
            #fact_text = f"- {row['subject']} {row['predicate']} {row['object']}" 
            if pd.notna(page):
                page = int(page)
                if page not in contexts[key]:
                    contexts[key].append(page)
                    pages_cited.append(page)

        for (subj, pred, obj), pages in contexts.items():
            verb = predicate_to_verb(pred)
            if pred_conversion:
                base = f"- {subj} {verb} {obj}"
            else:
                base = f"- {subj} {pred} {obj}"
            if pages:
                page_text = ", ".join(str(p) for p in sorted(pages))
                base += f" (Page {page_text})"
            context_parts.append(base)
        
        # Optionally retrieve original text chunks
        if use_chunks:
            if raw_texts:                                       # PROBLEMA: capire come funziona questa cosa
                text_chunks = get_text_context(pages_cited)
                if text_chunks:
                    context_parts.append("\nRelevant Text Excerpts:")
                    for chunk in text_chunks:
                        excerpt = chunk.get('text', '')[:200]  # Limit length
                        page = chunk.get('page', 'N/A')
                        context_parts.append(f"[Page {page}] {excerpt}...")
    
    context = "\n".join(context_parts)
    # print(f"CONTEXT:\n{context}\n")
    # 4. Build RAG prompt
    prompt = f"""Based on the following information from a food science handbook:

{context}

Please answer this question: {question}

Provide a clear answer and cite the page numbers where the information comes from."""
    
    # 5. Generate with LLM
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = f"Error generating response: {e}"
    
    latency = time.time() - start_time
    
    metadata = {
        "method": "RAG-KG",
        "question": question,
        "retrieved_facts": retrieved_facts.to_dict('records') if not retrieved_facts.empty else [],
        "context": context,
        "context_length": len(prompt),
        "latency_seconds": latency,
        "pages_cited": sorted(set(pages_cited))
    }
    
    return answer, metadata

print("RAG-KG baseline function defined")

RAG-KG baseline function defined


In [7]:
def zero_shot_baseline(question, model_name="groq/compound"):
    """
    Zero-shot LLM baseline.
    Sends question directly to LLM without any KG retrieval or grounding.
    
    Returns:
        - answer: Generated text
        - metadata: dict with latency, context_length
    """
    start_time = time.time()
    
    # Simple prompt with no additional context
    prompt = f"Answer this question: {question}"
    
    # Generate with LLM
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = f"Error generating response: {e}"
    
    latency = time.time() - start_time
    
    metadata = {
        "method": "Zero-Shot",
        "question": question,
        "context": None,
        "context_length": len(prompt),
        "latency_seconds": latency,
        "pages_cited": []
    }
    
    return answer, metadata

print("Zero-shot baseline function defined")

Zero-shot baseline function defined


In [8]:
# Load test data
test_data = []
with open('data/train/test/val/test_instructions.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_data.append(json.loads(line))

print(f"Loaded {len(test_data)} test samples")
print(f"Sample test item: {test_data[0]['instruction']}")

Loaded 231 test samples
Sample test item: What official advice mentions vegetables?


In [9]:
train_value = 2

save_dir = os.path.join("models", "baselines", f"train{train_value}")
os.makedirs(save_dir, exist_ok=True)

rag_save_path = os.path.join(save_dir, "rag_predictions.jsonl")
zero_shot_save_path = os.path.join(save_dir, "zero_shot_predictions.jsonl")

# Open JSONL files in append mode
rag_file = open(rag_save_path, "w", encoding="utf-8")
zero_file = open(zero_shot_save_path, "w", encoding="utf-8")

In [10]:
import time
import json
from math import ceil

batch_size = 20
num_batches = ceil(len(test_data) / batch_size)

for b in range(num_batches):
    start = b * batch_size
    end = start + batch_size
    batch = test_data[start:end]

    print(f"Processing batch {b+1}/{num_batches} ({len(batch)} items)...")

    for item in batch:
        instruction = item["instruction"]

        pred_ragkg, rag_meta = rag_kg_baseline(instruction)
        rag_file.write(json.dumps({"instruction": instruction, "prediction": pred_ragkg, "metadata": rag_meta}) + "\n")

        pred_zero, zero_meta = zero_shot_baseline(instruction)
        zero_file.write(json.dumps({"instruction": instruction, "prediction": pred_zero, "metadata": zero_meta}) + "\n")

    # Wait 1 minute unless it's the last batch
    if b < num_batches - 1:
        print("Waiting 60 seconds before next batch...")
        time.sleep(60)

rag_file.close()
zero_file.close()

print("RAG-KG and Zero-Shot LLM answers on test set saved")

Processing batch 1/12 (20 items)...


C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: D

Waiting 60 seconds before next batch...
Processing batch 2/12 (20 items)...


C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: D

Waiting 60 seconds before next batch...
Processing batch 3/12 (20 items)...


...

Waiting 60 seconds before next batch...
Processing batch 11/12 (20 items)...


C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: D

Waiting 60 seconds before next batch...
Processing batch 12/12 (11 items)...


C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined = combined.applymap(lambda x: str(x) if isinstance(x, list) else x)
C:\Users\salir\AppData\Local\Temp\ipykernel_10904\3156732290.py:54: FutureWarning: D

RAG-KG and Zero-Shot LLM answers on test set saved
